In [1]:
import re
from pathlib import Path
import pandas as pd
from PyPDF2 import PdfReader

VERIFY_FILE_RE = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2})_l(?P<level>\d+)_(?P<match_name>.+)_verify\.pdf$",
    re.I
)


# More permissive header regex:
# - DIV and CLASS can be empty
# - case-insensitive
SHOOTER_RE = re.compile(
    r'^(?P<bib>\d+)\s+(?P<name>.+?)\s+DIV:\s*(?P<div>[A-Z0-9]*)\s+'
    r'CLASS:\s*(?P<class>[A-Z0-9]*)\s+FACTOR:\s*(?P<pf>[A-Za-z]+)\s+'
    r'CATEGORY:\s*(?P<cat>.*)$',
    re.I
)

def _parse_shooter_line_fallback(line: str):
    """
    Fallback parser for slightly mangled headers.
    Expects:
      <bib> <name> DIV: ... CLASS: ... FACTOR: ... CATEGORY: ...
    but tolerates missing div/class/category values.
    """
    if not re.match(r'^\d+\s+', line):
        return None
    if "DIV:" not in line or "FACTOR:" not in line:
        return None

    try:
        # Split at DIV:
        left, right = line.split("DIV:", 1)
        left = left.strip()
        bib = left.split()[0]
        name = left[len(bib):].strip()

        # Now parse labeled sections in order
        div_part, rest = right.split("CLASS:", 1) if "CLASS:" in right else (right, "")
        div = div_part.strip()

        class_part, rest2 = rest.split("FACTOR:", 1) if "FACTOR:" in rest else ("", rest)
        class_ = class_part.strip()

        pf_part, cat_part = rest2.split("CATEGORY:", 1) if "CATEGORY:" in rest2 else (rest2, "")
        pf = pf_part.strip().split()[0] if pf_part.strip() else ""
        cat = cat_part.strip()

        return {
            "shooter": name,
            "div": div,
            "class": class_,
            "power_factor": pf,
            "cat": cat
        }
    except Exception:
        return None

def _parse_stage_line(line: str):
    toks = line.split()
    if len(toks) < 4 or toks[0] != "Stage":
        return None

    # We assume the canonical order is:
    # Stage N FACTOR PTS A C D Ded. MI NS PE OT Time
    # But we’ll be defensive and read from the front and end.

    stg = toks[1]
    factor = toks[2] if len(toks) > 2 else ""
    pts = toks[3] if len(toks) > 3 else ""
    a = toks[4] if len(toks) > 4 else ""
    c = toks[5] if len(toks) > 5 else ""
    d = toks[6] if len(toks) > 6 else ""

    # Whatever remains tends to include DED/MI/NS/PE/(OT)/Time.
    rest = toks[7:]

    # Time is almost always last token
    time = rest[-1] if rest else ""

    # Try to map the 4 penalty tokens if present
    ded = rest[0] if len(rest) > 0 else ""
    mi  = rest[1] if len(rest) > 1 else ""
    ns  = rest[2] if len(rest) > 2 else ""
    pe  = rest[3] if len(rest) > 3 else ""

    ot = ""  # OT is not reliably separated in extracted text

    return dict(
        stg=stg, factor=factor, pts=pts, a=a, c=c, d=d,
        ded=ded, mi=mi, ns=ns, pe=pe, ot=ot, time=time
    )

def _try_parse_shooter(line: str):
    m = SHOOTER_RE.match(line)
    if m:
        return {
            "shooter": m.group("name").strip(),
            "div": m.group("div").strip(),
            "class": m.group("class").strip(),
            "power_factor": m.group("pf").strip(),
            "cat": m.group("cat").strip(),
        }
    # Fallback label-based parse
    return _parse_shooter_line_fallback(line)

def parse_verify_pdf_text(pdf_path: str | Path) -> pd.DataFrame:
    reader = PdfReader(str(pdf_path))
    rows = []
    current = None
    header_buf = None

    for page in reader.pages:
        text = page.extract_text() or ""
        for raw_line in text.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            # If previous line looked like a partial header, try combining
            if header_buf:
                combined = f"{header_buf} {line}".strip()
                shooter = _try_parse_shooter(combined)
                if shooter:
                    current = shooter
                    header_buf = None
                    continue
                # If still no success, keep going (and drop buffer)
                header_buf = None

            shooter = _try_parse_shooter(line)
            if shooter:
                current = shooter
                continue

            # Heuristic: header line that got wrapped before FACTOR/CATEGORY
            if re.match(r'^\d+\s+.+\s+DIV:\s*', line, flags=re.I) and "FACTOR:" not in line:
                header_buf = line
                continue

            if line.startswith("Stage") and current:
                st = _parse_stage_line(line)
                if st:
                    rows.append({**current, **st})

    return pd.DataFrame(rows)

def get_match_data_text(match_file: str | Path) -> pd.DataFrame:
    path = Path(match_file)
    m = VERIFY_FILE_RE.search(path.name)

    if m:
        match_date = m.group("date")
        match_level = m.group("level")
        raw_name = m.group("match_name")

        # Build a normalized composite name if you want
        match_name = f"{match_date}_L{match_level}_{raw_name}"
    else:
        # Fallback if filename doesn't follow the convention
        match_date = path.name[:10]
        match_level = ""
        # everything before .pdf
        match_name = path.stem.replace("_verify", "")

    df = parse_verify_pdf_text(path)
    if df.empty:
        return df

    df["match_name"] = raw_name
    df["match_date"] = match_date
    df["match_level"] = match_level

    for col in ["factor", "time"]:
        df[col] = (
            df[col].astype(str)
                  .str.replace(",", ".", regex=False)
                  .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["pts"] = pd.to_numeric(df["pts"], errors="coerce").fillna(0).astype(int)
    # df["hit_factor"] = (df["pts"] / df["time"]).round(4)

    cols = [
        "match_name", "match_level", "match_date",
        "shooter", "div", "class", "power_factor", "cat",
        "stg", "factor",  "time", "pts", "a", "c", "d",
        "ded", "mi", "ns", "pe", "ot",
    ]
    df = df[[c for c in cols if c in df.columns]]
    cols = [
        "match_name", "match_level", "match_date",
        "shooter_name", "shooter_div", "shooter_class", "shooter_pf", "shooter_cat",
        "stg_n", "stg_hf", "stg_time", "stg_pts", "stg_a", "stg_c", "stg_d",
        "stg_ded", "stg_mi", "stg_ns", "stg_pe", "stg_ot",
    ]
    df.columns = cols
    return df


def collect_matches_text(pdf_dir: str | Path) -> pd.DataFrame:
    pdf_dir = Path(pdf_dir)
    files = sorted(pdf_dir.rglob("*_verify.pdf"))

    out = []
    for f in files:
        df = get_match_data_text(f)
        if not df.empty:
            out.append(df)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


In [2]:
df = collect_matches_text("pdf")

In [4]:
df[df["shooter_name"].str.contains("BUZZELLI, ANTONIO", na=False)]
# [
#     ["match_name", "shooter", "div", "class", "stg", "pts", "time", "factor"]
# ]

,match_name,match_level,match_date,shooter_name,shooter_div,shooter_class,shooter_pf,shooter_cat,stg_n,stg_hf,stg_time,stg_pts,stg_a,stg_c,stg_d,stg_ded,stg_mi,stg_ns,stg_pe,stg_ot
1216,ma5_2,2,2025-02-23,"BUZZELLI, ANTONIO",P,D,Minor,,1,3.9229,29.57,116,23,0,1,0,0,0,0,
1217,ma5_2,2,2025-02-23,"BUZZELLI, ANTONIO",P,D,Minor,,2,1.1946,34.32,41,9,2,0,0,1,0,0,
1218,ma5_2,2,2025-02-23,"BUZZELLI, ANTONIO",P,D,Minor,,3,4.4300,33.86,150,27,5,0,0,0,0,0,
1219,ma5_2,2,2025-02-23,"BUZZELLI, ANTONIO",P,D,Minor,,4,4.9394,22.27,110,20,3,1,0,0,0,0,
1220,ma5_2,2,2025-02-23,"BUZZELLI, ANTONIO",P,D,Minor,,5,3.9242,14.78,58,11,1,0,0,0,0,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27459,w4_5,2,2025-11-30,"BUZZELLI, ANTONIO",P,D,Minor,,2,4.1958,12.87,54,9,3,0,0,0,0,0,
27460,w4_5,2,2025-11-30,"BUZZELLI, ANTONIO",P,D,Minor,,3,5.6229,20.63,116,22,2,0,0,0,0,0,
27461,w4_5,2,2025-11-30,"BUZZELLI, ANTONIO",P,D,Minor,,4,6.2284,8.67,54,9,3,0,0,0,0,0,
27462,w4_5,2,2025-11-30,"BUZZELLI, ANTONIO",P,D,Minor,,5,6.5336,16.53,108,18,6,0,0,0,0,0,


In [ ]:
stg_shooter = df.groupby(['match_name', 'shooter'])['stg'].count().reset_index()
match_stg_median = stg_shooter.groupby('match_name')['stg'].median()
for match, median in match_stg_median.items():
    match_df = df[df['match_name'] == match]
    shooters = match_df['shooter'].unique()
    for shooter in shooters:
        shooter_df = match_df[match_df['shooter'] == shooter]
        count_stg = shooter_df.shape[0]
        if count_stg != median:
            print(match, shooter)